In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LinearRegression

# -------------------------
# Load
# -------------------------
train = pd.read_csv("GDP_Forecast_train.csv")
test  = pd.read_csv("GDP_Forecast_test.csv")
pmi   = pd.read_csv("US ISM Manufacturing PMI -31-01-1989-31-12-2024.csv")

# -------------------------
# GDP quarter handling
# -------------------------
def yq_to_period(yq: str) -> pd.Period:
    m = re.fullmatch(r"(\d{4})Q([1-4])", str(yq).strip())
    if not m:
        raise ValueError(f"Bad YQ format: {yq}")
    return pd.Period(year=int(m.group(1)), quarter=int(m.group(2)), freq="Q")

train["period"] = train["YQ"].apply(yq_to_period)
test["period"]  = test["YQ"].apply(yq_to_period)
train["qend"] = train["period"].dt.to_timestamp(how="end").dt.normalize()
test["qend"]  = test["period"].dt.to_timestamp(how="end").dt.normalize()

# -------------------------
# PMI parsing + quarterly average
# -------------------------
pmi["Date"]  = pd.to_datetime(pmi["Date"], errors="coerce", dayfirst=True)
pmi["Value"] = pd.to_numeric(pmi["Value"], errors="coerce")
pmi = pmi.dropna(subset=["Date","Value"]).copy()

pmi["period"] = pmi["Date"].dt.to_period("Q")
pmi_q = pmi.groupby("period")["Value"].mean().reset_index()
pmi_q = pmi_q.rename(columns={"Value":"PMI"})
pmi_q["qend"] = pmi_q["period"].dt.to_timestamp(how="end").dt.normalize()

# -------------------------
# Merge PMI into train/test
# -------------------------
train = train.merge(pmi_q[["qend","PMI"]], on="qend", how="left")
test  = test.merge(pmi_q[["qend","PMI"]], on="qend", how="left")

# -------------------------
# Build full timeline and create lags ONCE
# -------------------------
full = pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True)
full = full.sort_values("qend").reset_index(drop=True)

full["PMI_lag1"] = full["PMI"].shift(1)
full["PMI_lag2"] = full["PMI"].shift(2)

# -------------------------
# Split by original source, not by NGDP values (test NGDP may be 0 placeholders)
# -------------------------
train_model = full[full["_split"]=="train"].copy()
test_model  = full[full["_split"]=="test"].copy()

# Drop rows that cannot have lags
train_model = train_model.dropna(subset=["PMI","PMI_lag1","PMI_lag2","NGDP"]).copy()
test_model  = test_model.dropna(subset=["PMI","PMI_lag1","PMI_lag2"]).copy()

# -------------------------
# Fit
# -------------------------
X = train_model[["PMI","PMI_lag1","PMI_lag2"]].astype(float)
y = train_model["NGDP"].astype(float)

print("Train usable rows:", len(train_model))
print("Test rows to predict:", len(test_model))

model = LinearRegression()
model.fit(X, y)

# -------------------------
# Predict
# -------------------------
X_test = test_model[["PMI","PMI_lag1","PMI_lag2"]].astype(float)
pred = model.predict(X_test)

out = test_model[["YQ"]].copy()
out["NGDP"] = pred.round(1)

out.to_csv("PMI_forecast_submission.csv", index=False)

print(out.head())

Train usable rows: 102
Test rows to predict: 20
         YQ  NGDP
104  2016Q1   4.4
105  2016Q2   4.7
106  2016Q3   3.9
107  2016Q4   5.4
108  2017Q1   6.7


C:\Users\nghoc\AppData\Local\Temp\ipykernel_30120\891187719.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pmi["Date"]  = pd.to_datetime(pmi["Date"], errors="coerce", dayfirst=True)


In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# If you don't have xgboost: pip install xgboost
import xgboost as xgb

# ----------------------------
# 0) Paths (adjust if needed)
# ----------------------------
PATH_TRAIN = "GDP_Forecast_train.csv"
PATH_TEST  = "GDP_Forecast_test.csv"

PATH_UNRATE = "UNRATE.csv"
PATH_CPI    = "CPIAUCSL.csv"
PATH_DFF    = "DFF.csv"
PATH_DSPI   = "DSPI.csv"
PATH_PPI    = "PPIACO.csv"
PATH_EXR    = "Real Broad Effective Exchange Rate-RBUSBIS-01-01-1994-31-12-2024.csv"
PATH_PMI_M  = "US ISM Manufacturing PMI -31-01-1989-31-12-2024.csv"
PATH_PMI_S  = "US ISM Services PMI- 31-03-2008-31-12-2024.csv"

# ----------------------------
# 1) Helpers
# ----------------------------
def yq_to_period(yq: str) -> pd.Period:
    m = re.fullmatch(r"(\d{4})Q([1-4])", str(yq).strip())
    if not m:
        raise ValueError(f"Bad YQ format: {yq}")
    return pd.Period(year=int(m.group(1)), quarter=int(m.group(2)), freq="Q")

def load_time_series(path, date_col=None, value_col=None, name=None, dayfirst=False):
    df = pd.read_csv(path)
    if date_col is None:
        for c in df.columns:
            if c.lower() in ["date", "observation_date", "time", "timestamp"]:
                date_col = c
                break
    if value_col is None:
        if "Value" in df.columns:
            value_col = "Value"
        else:
            non_date = [c for c in df.columns if c != date_col]
            value_col = non_date[0]
    if name is None:
        name = value_col

    out = df[[date_col, value_col]].copy()
    out = out.rename(columns={date_col: "date", value_col: name})
    out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=dayfirst)
    out[name] = pd.to_numeric(out[name], errors="coerce")
    out = out.dropna(subset=["date", name]).sort_values("date")
    return out

def to_quarterly_mean(ts_df, colname):
    tmp = ts_df.copy()
    tmp["period"] = tmp["date"].dt.to_period("Q")
    q = tmp.groupby("period")[colname].mean().to_frame()
    q["qend"] = q.index.to_timestamp(how="end").normalize()
    q = q.set_index("qend")[[colname]].sort_index()
    return q

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# ----------------------------
# 2) Load GDP target data
# ----------------------------
train = pd.read_csv(PATH_TRAIN)
test  = pd.read_csv(PATH_TEST)

train["period"] = train["YQ"].apply(yq_to_period)
test["period"]  = test["YQ"].apply(yq_to_period)

train["qend"] = train["period"].dt.to_timestamp(how="end").dt.normalize()
test["qend"]  = test["period"].dt.to_timestamp(how="end").dt.normalize()

# Keep source flag to avoid the "test NGDP=0" split problem
train["_split"] = "train"
test["_split"]  = "test"

# ----------------------------
# 3) Load macro series and convert to quarterly
# ----------------------------
series_specs = [
    (PATH_UNRATE, None, None, "UNRATE", False),
    (PATH_CPI,    None, None, "CPI",    False),
    (PATH_DFF,    None, None, "DFF",    False),   # daily -> quarterly mean
    (PATH_DSPI,   None, None, "DSPI",   False),
    (PATH_PPI,    None, None, "PPI",    False),
    (PATH_EXR,    None, None, "RBUSBIS",False),

    # PMI files often use "31-Dec-24" style; dayfirst=True is safer
    (PATH_PMI_M,  "Date", "Value", "PMI_M", True),
    (PATH_PMI_S,  "Date", "Value", "PMI_S", True),
]

qdfs = []
for path, dcol, vcol, name, dayfirst in series_specs:
    ts = load_time_series(path, date_col=dcol, value_col=vcol, name=name, dayfirst=dayfirst)
    qdfs.append(to_quarterly_mean(ts, name))

macro_q = pd.concat(qdfs, axis=1).sort_index()

# ----------------------------
# 4) Build a master quarterly frame over train+test horizon
# ----------------------------
all_periods = pd.period_range(train["period"].min(), test["period"].max(), freq="Q")
master = pd.DataFrame({"period": all_periods})
master["qend"] = master["period"].dt.to_timestamp(how="end").dt.normalize()
master = master.set_index("qend").sort_index()

df = master.join(macro_q, how="left")

# Attach GDP train labels by quarter
df = df.join(train.set_index("qend")[["NGDP"]], how="left")

# Attach split flags by quarter (train/test)
full_split = pd.concat([train[["qend","YQ","_split"]], test[["qend","YQ","_split"]]], ignore_index=True)
full_split = full_split.drop_duplicates(subset=["qend"]).set_index("qend")
df = df.join(full_split, how="left")

# ----------------------------
# 5) Feature engineering
# ----------------------------
macro_cols = ["UNRATE","CPI","DFF","DSPI","PPI","RBUSBIS","PMI_M","PMI_S"]
feat = df.copy()

# time features
feat["year"] = feat["period"].dt.year.astype(int)
feat["qtr"]  = feat["period"].dt.quarter.astype(int)

# macro transforms
for c in macro_cols:
    if c in feat.columns:
        feat[f"{c}_qoq_pct"] = feat[c].pct_change(1) * 100
        feat[f"{c}_yoy_pct"] = feat[c].pct_change(4) * 100
        feat[f"{c}_diff1"]   = feat[c].diff(1)
        feat[f"{c}_diff4"]   = feat[c].diff(4)

# autoregressive target lags (very important)
for lag in [1,2,3,4]:
    feat[f"NGDP_lag{lag}"] = feat["NGDP"].shift(lag)

# ----------------------------
# 6) Train/Validation split (time-based)
# ----------------------------
# Only keep quarters belonging to original train/test
train_frame = feat[feat["_split"]=="train"].copy()
test_frame  = feat[feat["_split"]=="test"].copy()

# Drop rows where the label is missing (should not happen for train)
train_frame = train_frame[train_frame["NGDP"].notna()].copy()

# Feature columns (exclude non-features)
non_features = {"period","NGDP","YQ","_split"}
feature_cols = sorted([c for c in feat.columns if c not in non_features])

X_all = train_frame[feature_cols]
y_all = train_frame["NGDP"].astype(float)

# last 12 quarters as validation (adjust if you want)
n_val = 12 if len(train_frame) > 40 else max(4, len(train_frame)//4)
X_tr, y_tr = X_all.iloc[:-n_val], y_all.iloc[:-n_val]
X_val, y_val = X_all.iloc[-n_val:], y_all.iloc[-n_val:]

# ----------------------------
# 7) Models
# ----------------------------
ridge = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("model", Ridge(alpha=10.0, random_state=42))
])

rf = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=1200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

xgbr = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("model", xgb.XGBRegressor(
        n_estimators=4000,
        learning_rate=0.02,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

models = {"Ridge": ridge, "RF": rf, "XGB": xgbr}

# Fit and validate
val_rmse = {}
val_pred = {}

for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_val)
    val_rmse[name] = rmse(y_val, p)
    val_pred[name] = p

print("Validation RMSE (lower is better):")
for k, v in val_rmse.items():
    print(f"  {k}: {v:.4f}")

# RMSE-weighted ensemble (inverse RMSE)
inv = {k: 1.0/max(v, 1e-9) for k, v in val_rmse.items()}
s = sum(inv.values())
w = {k: inv[k]/s for k in inv}
print("Ensemble weights:")
for k, v in w.items():
    print(f"  {k}: {v:.3f}")

# ----------------------------
# 8) Refit on ALL training data, predict test, ensemble
# ----------------------------
X_test = test_frame[feature_cols]

test_preds = {}
for name, m in models.items():
    m.fit(X_all, y_all)
    test_preds[name] = m.predict(X_test)

ensemble_pred = sum(w[name] * test_preds[name] for name in models.keys())

# ----------------------------
# 9) Output submission (1 decimal)
# ----------------------------
submission = test_frame[["YQ"]].copy()
submission["NGDP"] = np.round(ensemble_pred, 1)

submission.to_csv("FINAL_macro_xgb_ensemble.csv", index=False, float_format="%.1f")
print("Saved: FINAL_macro_xgb_ensemble.csv")
print(submission.head(10))

C:\Users\nghoc\AppData\Local\Temp\ipykernel_30120\4175786290.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=dayfirst)
C:\Users\nghoc\AppData\Local\Temp\ipykernel_30120\4175786290.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=dayfirst)


Validation RMSE (lower is better):
  Ridge: 2.9877
  RF: 2.1939
  XGB: 2.1450
Ensemble weights:
  Ridge: 0.266
  RF: 0.363
  XGB: 0.371
Saved: FINAL_macro_xgb_ensemble.csv
                YQ  NGDP
qend                    
2016-03-31  2016Q1   4.7
2016-06-30  2016Q2   4.0
2016-09-30  2016Q3   3.4
2016-12-31  2016Q4   5.2
2017-03-31  2017Q1   6.3
2017-06-30  2017Q2   5.4
2017-09-30  2017Q3   6.3
2017-12-31  2017Q4   5.3
2018-03-31  2018Q1   5.5
2018-06-30  2018Q2   5.5
